# OpenAI GPT-2 pretrained weights

This notebook loads GPT-2 small (124M) weights into the custom PyTorch implementation in this repository. It uses the original GPT-2 BPE tokenizer. The original character-level training notebook remains a separate experiment.

Reference: [Raschka’s weight-loading chapter](https://github.com/rasbt/LLMs-from-scratch/tree/main/ch05/02_alternative_weight_loading). GPT-2 generates text continuations; it is not an instruction-tuned chatbot.


In [ ]:
!git clone https://github.com/raoankit72005/minigpt.git /content/minigpt
%cd /content/minigpt
%pip install -r requirements.txt


## Load and verify
Choose a GPU runtime for faster generation; CPU also works. The first run downloads roughly 500 MB of model weights plus the tokenizer. No paid API key is needed. The loader verifies its logits against the downloaded reference before releasing that model.


In [ ]:
import torch
from gpt2_pretrained import load_pretrained
device = "cuda" if torch.cuda.is_available() else "cpu"
model, tokenizer = load_pretrained(cache_dir="checkpoints")
model = model.to(device)
print("Device:", device)
print("Parameters:", sum(p.numel() for p in model.parameters()))


## Generate text
Change PROMPT and run this cell again. Sampling uses temperature and top-k; the full prompt plus its continuation is printed.


In [ ]:
PROMPT = "Every effort moves you"
ids = tokenizer.encode(PROMPT, return_tensors="pt").to(device)
if ids.shape[1] == 0:
    ids = torch.tensor([[tokenizer.eos_token_id]], device=device)
torch.manual_seed(42)
output = model.generate(ids, max_new_tokens=100, temperature=0.8, top_k=40, eos_id=tokenizer.eos_token_id)
print(tokenizer.decode(output[0].tolist(), skip_special_tokens=True))


## Optional: save converted weights
The Hugging Face cache already retains the downloaded weights for the lifetime of this runtime. This cell saves a portable custom checkpoint. Colab storage is temporary; copy the checkpoint and tokenizer to Drive if you want to retain them. Do not commit these large files to Git.


In [ ]:
from dataclasses import asdict
from pathlib import Path
OUT = Path("checkpoints/converted_gpt2")
OUT.mkdir(parents=True, exist_ok=True)
torch.save({"config": asdict(model.cfg), "model_id": "openai-community/gpt2",
            "state_dict": {k: v.cpu() for k, v in model.state_dict().items()}}, OUT / "gpt2.pt")
tokenizer.save_pretrained(OUT)
print("Saved:", OUT)


## Optional: restore the converted checkpoint
Run after imports/installation when the saved directory is available.


In [ ]:
from gpt2_pretrained import GPT2, GPT2Config
from transformers import GPT2TokenizerFast
checkpoint = torch.load(OUT / "gpt2.pt", map_location="cpu", weights_only=True)
restored = GPT2(GPT2Config(**checkpoint["config"]))
restored.load_state_dict(checkpoint["state_dict"], strict=True)
restored = restored.to(device).eval()
restored_tokenizer = GPT2TokenizerFast.from_pretrained(OUT)
